### Imports and MLflow setup

In [32]:
# from mlflow.tracking import MlflowClient

# client = MlflowClient()

# # Find the deleted experiment
# experiment = client.get_experiment_by_name("employee-attrition")
# print(experiment.experiment_id)   # note this ID

# # Restore it
# client.restore_experiment(experiment.experiment_id)

# # # Now this works again
# # mlflow.set_experiment("employee-attrition")

In [33]:
import os
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

mlflow.set_experiment("employee-attrition")
# optional: mlflow.set_tracking_uri("file:///absolute/path/to/mlruns")

<Experiment: artifact_location='file:///d:/employee-attrition-mlops/mlruns/1', creation_time=1781110083006, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1781217756037, lifecycle_stage='active', name='employee-attrition', tags={}, trace_location=None, workspace='default'>

### Load data + simple split

In [34]:

employee_df = pd.read_csv("EmployeeAttrition.csv")  # or data/train.csv


In [35]:
categorical_cols = employee_df.select_dtypes(include=['object', 'category']).columns
# # new df contains only categorical columns
employee_df_categorical = employee_df[categorical_cols]


In [36]:
categorical_cols

Index(['Gender', 'Job Role', 'Work-Life Balance', 'Job Satisfaction',
       'Performance Rating', 'Overtime', 'Education Level', 'Marital Status',
       'Job Level', 'Company Size', 'Remote Work', 'Leadership Opportunities',
       'Innovation Opportunities', 'Company Reputation',
       'Employee Recognition', 'Attrition'],
      dtype='object')

In [37]:
employee_df_categorical

,Gender,Job Role,Work-Life Balance,Job Satisfaction,Performance Rating,Overtime,Education Level,Marital Status,Job Level,Company Size,Remote Work,Leadership Opportunities,Innovation Opportunities,Company Reputation,Employee Recognition,Attrition
0,Male,Education,Excellent,Medium,Average,No,Associate Degree,Married,Mid,Medium,No,No,No,Excellent,Medium,Stayed
1,Female,Media,Poor,High,Low,No,Master’s Degree,Divorced,Mid,Medium,No,No,No,Fair,Low,Stayed
2,Female,Healthcare,Good,High,Low,No,Bachelor’s Degree,Married,Mid,Medium,No,No,No,Poor,Low,Stayed
3,Female,Education,Good,High,High,No,High School,Single,Mid,Small,Yes,No,No,Good,Medium,Stayed
4,Male,Education,Fair,Very High,Average,Yes,High School,Divorced,Senior,Medium,No,No,No,Fair,Medium,Stayed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74493,Female,Healthcare,Poor,Medium,Average,Yes,Associate Degree,Single,Senior,Medium,No,No,No,Poor,Medium,Stayed
74494,Female,Education,Good,Medium,Average,Yes,Master’s Degree,Married,Entry,Medium,No,No,No,Good,Medium,Left
74495,Male,Education,Good,Very High,Below Average,No,Associate Degree,Married,Mid,Small,No,No,No,Good,High,Left
74496,Male,Education,Fair,High,Average,No,Bachelor’s Degree,Divorced,Mid,Large,No,No,No,Poor,High,Stayed


In [38]:

# Step 1 — run your block as is (keep everything)
employee_df_encoded = employee_df.copy()
label_encoders = {}
label_encoded_data = []

for col in employee_df_encoded.select_dtypes(include=['object', 'category']).columns:
    le = LabelEncoder()
    employee_df_encoded[col] = le.fit_transform(employee_df_encoded[col])
    label_encoders[col] = le
    category_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    label_encoded_data.append(category_mapping)



In [39]:
label_encoded_data

[{'Female': np.int64(0), 'Male': np.int64(1)},
 {'Education': np.int64(0),
  'Finance': np.int64(1),
  'Healthcare': np.int64(2),
  'Media': np.int64(3),
  'Technology': np.int64(4)},
 {'Excellent': np.int64(0),
  'Fair': np.int64(1),
  'Good': np.int64(2),
  'Poor': np.int64(3)},
 {'High': np.int64(0),
  'Low': np.int64(1),
  'Medium': np.int64(2),
  'Very High': np.int64(3)},
 {'Average': np.int64(0),
  'Below Average': np.int64(1),
  'High': np.int64(2),
  'Low': np.int64(3)},
 {'No': np.int64(0), 'Yes': np.int64(1)},
 {'Associate Degree': np.int64(0),
  'Bachelor’s Degree': np.int64(1),
  'High School': np.int64(2),
  'Master’s Degree': np.int64(3),
  'PhD': np.int64(4)},
 {'Divorced': np.int64(0), 'Married': np.int64(1), 'Single': np.int64(2)},
 {'Entry': np.int64(0), 'Mid': np.int64(1), 'Senior': np.int64(2)},
 {'Large': np.int64(0), 'Medium': np.int64(1), 'Small': np.int64(2)},
 {'No': np.int64(0), 'Yes': np.int64(1)},
 {'No': np.int64(0), 'Yes': np.int64(1)},
 {'No': np.int64(0

In [40]:
# Step 2 — from this one base, create two versions
employee_df_tree = employee_df_encoded.copy()        # RF, XGBoost — done

In [41]:
#removing first Unnamed column
employee_df_tree = employee_df_tree.drop(columns=['Unnamed: 0'])
employee_df_tree

,Employee ID,Age,Gender,Years at Company,Job Role,Monthly Income,Work-Life Balance,Job Satisfaction,Performance Rating,Number of Promotions,...,Number of Dependents,Job Level,Company Size,Company Tenure,Remote Work,Leadership Opportunities,Innovation Opportunities,Company Reputation,Employee Recognition,Attrition
0,8410,31,1,19,0,5390,0,2,0,2,...,0,1,1,89,0,0,0,0,2,1
1,64756,59,0,4,3,5534,3,0,3,3,...,3,1,1,21,0,0,0,1,1,1
2,30257,24,0,10,2,8159,2,0,3,0,...,3,1,1,74,0,0,0,3,1,1
3,65791,36,0,7,0,3989,2,0,2,1,...,2,1,2,50,1,0,0,2,2,1
4,65026,56,1,41,0,4821,1,3,0,0,...,0,2,1,68,0,0,0,1,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74493,16243,56,0,42,2,7830,3,2,0,0,...,0,2,1,60,0,0,0,3,2,1
74494,47175,30,0,15,0,3856,2,2,0,2,...,0,0,1,20,0,0,0,2,2,0
74495,12409,52,1,5,0,5654,2,3,1,0,...,4,1,2,7,0,0,0,2,0,0
74496,9554,18,1,4,0,5276,1,0,0,0,...,3,1,0,5,0,0,0,3,0,1


In [42]:
employee_df_linear = pd.get_dummies(                 # LR — fix Marital Status on top
    employee_df_encoded.copy(),
    columns=["Marital Status"],
    drop_first=True
)

In [43]:
employee_df_linear

,Unnamed: 0,Employee ID,Age,Gender,Years at Company,Job Role,Monthly Income,Work-Life Balance,Job Satisfaction,Performance Rating,...,Company Size,Company Tenure,Remote Work,Leadership Opportunities,Innovation Opportunities,Company Reputation,Employee Recognition,Attrition,Marital Status_1,Marital Status_2
0,0,8410,31,1,19,0,5390,0,2,0,...,1,89,0,0,0,0,2,1,True,False
1,1,64756,59,0,4,3,5534,3,0,3,...,1,21,0,0,0,1,1,1,False,False
2,2,30257,24,0,10,2,8159,2,0,3,...,1,74,0,0,0,3,1,1,True,False
3,3,65791,36,0,7,0,3989,2,0,2,...,2,50,1,0,0,2,2,1,False,True
4,4,65026,56,1,41,0,4821,1,3,0,...,1,68,0,0,0,1,2,1,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74493,74493,16243,56,0,42,2,7830,3,2,0,...,1,60,0,0,0,3,2,1,False,True
74494,74494,47175,30,0,15,0,3856,2,2,0,...,1,20,0,0,0,2,2,0,True,False
74495,74495,12409,52,1,5,0,5654,2,3,1,...,2,7,0,0,0,2,0,0,True,False
74496,74496,9554,18,1,4,0,5276,1,0,0,...,0,5,0,0,0,3,0,1,False,False


### Prepare train/test splits

In [44]:
from sklearn.model_selection import train_test_split

def prepare_dataset(df, target_col="Attrition", test_size=0.2, random_state=42):
    X = df.drop(columns=[target_col])
    y = df[target_col]
    if y.dtype == object:
        y = y.map({"Yes": 1, "No": 0})
    return train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

### Then split both datasets:

In [45]:
X_tree_train, X_tree_test, y_tree_train, y_tree_test = prepare_dataset(
    employee_df_tree,
    target_col="Attrition",
    test_size=0.2,
    random_state=42
)

X_linear_train, X_linear_test, y_linear_train, y_linear_test = prepare_dataset(
    employee_df_linear,
    target_col="Attrition",
    test_size=0.2,
    random_state=42
)

###  Log split metadata to MLflow

In [46]:
import mlflow

def log_split_metadata(dataset_name, X_train, X_test, y_train, y_test, test_size, random_state):
    with mlflow.start_run(run_name=f"{dataset_name}-split"):
        mlflow.set_tag("dataset_type", dataset_name)
        mlflow.log_param("test_size", test_size)
        mlflow.log_param("random_state", random_state)
        mlflow.log_param("n_rows_total", len(X_train) + len(X_test))
        mlflow.log_param("n_train_rows", len(X_train))
        mlflow.log_param("n_test_rows", len(X_test))
        mlflow.log_param("n_features", X_train.shape[1])
        mlflow.log_param("feature_names", ",".join(X_train.columns.astype(str)))
        mlflow.log_param("target_classes", ",".join(map(str, sorted(y_train.unique()))))

### Call it for each dataset

In [47]:
log_split_metadata(
    "tree",
    X_tree_train,
    X_tree_test,
    y_tree_train,
    y_tree_test,
    test_size=0.2,
    random_state=42
)

log_split_metadata(
    "linear",
    X_linear_train,
    X_linear_test,
    y_linear_train,
    y_linear_test,
    test_size=0.2,
    random_state=42
)

### Train and Log a model for each dataset

In [48]:
import mlflow
import mlflow.sklearn
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV


def train_and_log_model(run_name, dataset_name, model, params=None, param_grid=None, X_train=None, X_test=None, y_train=None, y_test=None, scoring="f1"):
    with mlflow.start_run(run_name=run_name):
        mlflow.set_tag("dataset_type", dataset_name)
        mlflow.set_tag("model_type", model.__class__.__name__)

        if param_grid:
            grid_search = GridSearchCV(
                estimator=model,
                param_grid=param_grid,
                cv=5,
                scoring=scoring,
                n_jobs=-1,
                verbose=1,
            )
            grid_search.fit(X_train, y_train)
            model = grid_search.best_estimator_
            mlflow.set_tag("grid_search", "true")
            mlflow.log_params(grid_search.best_params_)
            mlflow.log_metric("grid_search_best_score", grid_search.best_score_)
        else:
            if params:
                model.set_params(**params)
            model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1_score": f1_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred),
            "recall": recall_score(y_test, y_pred),
        }
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(model, "model")

        return metrics


### Example Usage

In [49]:
from sklearn.linear_model import LogisticRegression


tree_model = RandomForestClassifier(random_state=42)
tree_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "max_features": ["sqrt", "log2"],
    "criterion": ["gini", "entropy"],
}

linear_model = LogisticRegression(random_state=42, max_iter=1000)
linear_param_grid = {
    "C": [0.01, 0.1, 1.0, 10.0],
    "solver": ["liblinear", "saga"],
    "penalty": ["l2"],
    "max_iter": [1000],
}

train_and_log_model(
    run_name="rf-tree",
    dataset_name="tree",
    model=tree_model,
    param_grid=tree_param_grid,
    X_train=X_tree_train,
    X_test=X_tree_test,
    y_train=y_tree_train,
    y_test=y_tree_test,
    scoring="roc_auc"
)
train_and_log_model(
    run_name="logistic-linear",
    dataset_name="linear",
    model=linear_model,
    param_grid=linear_param_grid,
    X_train=X_linear_train,
    X_test=X_linear_test,
    y_train=y_linear_train,
    y_test=y_linear_test,
    scoring="f1"
)


Fitting 5 folds for each of 24 candidates, totalling 120 fits


2026/06/11 18:37:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/11 18:37:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Fitting 5 folds for each of 8 candidates, totalling 40 fits


2026/06/11 18:39:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/11 18:39:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


{'accuracy': 0.718523489932886,
 'f1_score': 0.7360271903323263,
 'precision': 0.7252542793351525,
 'recall': 0.7471249680552006}

In [50]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(random_state=42)
tree_param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

dt_metrics = train_and_log_model(
    run_name="decision-tree-tree",
    dataset_name="tree",
    model=tree_model,
    param_grid=tree_param_grid,
    X_train=X_tree_train,
    X_test=X_tree_test,
    y_train=y_tree_train,
    y_test=y_tree_test,
    scoring="f1"
)
print("Decision Tree metrics:", dt_metrics)


Fitting 5 folds for each of 72 candidates, totalling 360 fits


2026/06/11 18:41:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/11 18:41:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Decision Tree metrics: {'accuracy': 0.7265771812080537, 'f1_score': 0.739447429009977, 'precision': 0.7402048655569783, 'recall': 0.7386915410171224}


In [51]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)
xgb_param_grid = {
    "n_estimators": [100, 150],
    "max_depth": [4, 6],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}

xgb_metrics = train_and_log_model(
    run_name="xgboost-tree",
    dataset_name="tree",
    model=xgb_model,
    param_grid=xgb_param_grid,
    X_train=X_tree_train,
    X_test=X_tree_test,
    y_train=y_tree_train,
    y_test=y_tree_test,
    scoring = "roc_auc"
)
print("XGBoost metrics:", xgb_metrics)


Fitting 5 folds for each of 32 candidates, totalling 160 fits


c:\Users\aarun\anaconda3\envs\emp_att_mlops\lib\site-packages\xgboost\training.py:200: UserWarning: [18:41:50] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
2026/06/11 18:41:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/11 18:41:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


XGBoost metrics: {'accuracy': 0.7606711409395973, 'f1_score': 0.771878198567042, 'precision': 0.7728670253651038, 'recall': 0.7708918987988755}


In [52]:
from sklearn.neural_network import MLPClassifier

nn_model = MLPClassifier(random_state=42, max_iter=300)
nn_param_grid = {
    "hidden_layer_sizes": [(64, 32), (128, 64)],
    "activation": ["relu", "tanh"],
    "solver": ["adam"],
    "alpha": [0.0001, 0.001],
    "learning_rate_init": [0.001, 0.01],
    "max_iter": [300],
}

nn_metrics = train_and_log_model(
    run_name="mlp-linear",
    dataset_name="linear",
    model=nn_model,
    param_grid=nn_param_grid,
    X_train=X_linear_train,
    X_test=X_linear_test,
    y_train=y_linear_train,
    y_test=y_linear_test,
)
print("Neural Network metrics:", nn_metrics)


Fitting 5 folds for each of 16 candidates, totalling 80 fits


2026/06/11 18:51:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/11 18:51:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Neural Network metrics: {'accuracy': 0.525234899328859, 'f1_score': 0.6887265686878465, 'precision': 0.525234899328859, 'recall': 1.0}
